In [0]:
%python
df = spark.table("samples.accuweather.forecast_hourly_metric")
display(df)

In [0]:
%python
df = spark.read.csv("/Volumes/test_data/testing_data/test/employee.csv")
display(df)

In [0]:
%python
!pip install pandas

In [0]:
%python
import pandas as pd

dff = pd.read_csv("/Volumes/test_data/testing_data/test/employee.csv")

In [0]:
%python
dff.describe()

In [0]:
%python
import logging

logger = logging.getLogger("ingestion")
logger.setLevel(logging.INFO)

try:
    logger.info("Starting ingestion...")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/Volumes/test_data/testing_data/test/employee.csv")
    )

    logger.info(f"Rows loaded: {df.count()}")

    display(df)

except Exception as e:
    logger.exception("Ingestion failed")

In [0]:
%python
df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/Volumes/test_data/testing_data/test/employee_dataset_large.csv")
)

In [0]:
%python
df.show()

In [0]:
%python
df.dropDuplicates().show()

In [0]:
%python
df.filter(df.salary > 35000).show()

In [0]:
%python
from pyspark.sql.functions import avg
df.select(avg("salary")).show()


In [0]:
%python
df.count()

In [0]:
%python
df.printSchema()

In [0]:
%python
from pyspark.sql.functions import col

df = df.withColumn(
    "salary",
    col("salary").cast("int")
)

df.printSchema()

In [0]:
%python
df.printSchema()


In [0]:
%python
df.describe().show()

In [0]:
%python
df.filter(df["salary"] > 30000)              # filter rows
df.where(df["salary"] > 30000)  

In [0]:
%python
df.groupBy("department").count()

In [0]:
%python
df.orderBy("salary").show()

In [0]:
%sql
SELECT * FROM csv.`/Volumes/test_data/testing_data/test/employee_dataset_large.csv`

In [0]:
%sql

CREATE TEMP VIEW my_csv_view
USING csv
OPTIONS (
  path "/Volumes/test_data/testing_data/test/",
  header "true",
  inferSchema "true"
);

SELECT * FROM my_csv_view;

In [0]:
%sql

CREATE TEMP VIEW Employees_data
USING csv
OPTIONS (
  path "/Volumes/test_data/testing_data/test/employee_dataset_large.csv",
  header "true",
  inferSchema "true"
);

SELECT * FROM  Employees_data;

In [0]:
%sql
SELECT * FROM my_csv_view
where salary > 50000;

In [0]:

%sql

CREATE TEMP VIEW project_allocation
USING csv
OPTIONS (
  path "/Volumes/test_data/testing_data/test/project_allocations.csv",
  header "true",
  inferSchema "true"
);

SELECT * FROM project_allocation;

In [0]:
SELECT *
FROM project_allocation
LIMIT 5;

select * from Employees_data limit 5;

In [0]:
SELECT *
FROM Employees_data e
LEFT SEMI JOIN project_allocation p
ON e.emp_id = p.emp_id;

In [0]:
SELECT *
FROM Employees_data e
LEFT ANTI JOIN project_allocation p
ON e.emp_id = p.emp_id;

In [0]:
SELECT
CONCAT(name,' - ',department) AS employee_info
FROM employees_data;

In [0]:
SELECT
name,
LENGTH(name)
FROM employees_data;

In [0]:
SELECT
department,
REPLACE(department,'Sales','Marketing')
FROM employees_data;

In [0]:
SELECT
name,
DATEDIFF(CURRENT_DATE(),joining_date) AS days_worked
FROM employees_data;

In [0]:
SELECT
joining_date,
DATE_ADD(joining_date,30)
FROM employees_data;

In [0]:
SELECT
name,
DATEDIFF(CURRENT_DATE(),joining_date) AS days_worked
FROM employees_data;

In [0]:
SELECT
name,
salary,
SUM(salary)
OVER(ORDER BY joining_date)
AS running_total
FROM employees_data;

In [0]:
SELECT
department,
name,
salary,
SUM(salary)
OVER(
PARTITION BY department
ORDER BY joining_date
)
AS dept_running_total
FROM employees_data;

In [0]:
SELECT *
FROM (
SELECT *,
ROW_NUMBER() OVER(
PARTITION BY department
ORDER BY salary DESC
) rn
FROM employees_data
)
WHERE rn<=2;

In [0]:
SELECT *,
ROW_NUMBER() OVER(
PARTITION BY department
ORDER BY salary DESC
)  rn
FROM employees_data
QUALIFY rn=2;

In [0]:
SELECT *,
ROW_NUMBER() OVER(
PARTITION BY department
ORDER BY salary DESC
) rn
FROM employees_data
QUALIFY rn=1;

In [0]:
SELECT *
FROM
(
SELECT department,emp_id
FROM employees_data
)
PIVOT
(
COUNT(emp_id)
FOR department IN
('IT','HR','Finance','Sales','Legal')
);

In [0]:
CREATE VIEW test_data.testing_data.Employees_view AS
SELECT *
FROM csv.`/Volumes/test_data/testing_data/test/employee_dataset_large.csv`;

In [0]:
SELECT *
FROM
(
SELECT *,
ROW_NUMBER() OVER(
PARTITION BY emp_id
ORDER BY joining_date DESC
) rn
FROM ṇṇ
)
WHERE rn=1;

In [0]:
%sql
-- Employees with NO project allocation
SELECT e.*
FROM Employees_data e
LEFT ANTI JOIN project_allocation p
  ON e.emp_id = p.emp_id;

In [0]:
%sql
-- NOT EXISTS version
SELECT e.*
FROM Employees_data e
WHERE NOT EXISTS (
    SELECT 1 FROM project_allocation p
    WHERE p.emp_id = e.emp_id
);

In [0]:
%sql
-- Reverse: project allocations with no matching employee record
SELECT p.*
FROM project_allocation p
LEFT ANTI JOIN Employees_data e
  ON p.emp_id = e.emp_id;